# Session 5: Retrieval-Augmented Generation (RAG)

## Objectives
- Understand the RAG architecture and why it's important
- Implement document chunking
- Build a complete RAG pipeline from scratch
- Answer questions using custom knowledge

**Duration:** 40 minutes | **Level:** Medium

**Why RAG?** LLMs only know what they were trained on. RAG lets them answer questions about YOUR data — documents, databases, knowledge bases — without fine-tuning.

In [1]:
import os
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI

# Load environment variables from .env file
load_dotenv(dotenv_path=os.path.join("..", ".env"))

client = OpenAI()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")

print(f"Setup complete! Model: {MODEL}")

Setup complete! Model: gpt-4o-mini


## 1. The RAG Architecture

RAG has two main phases:

### Indexing Phase (done once)
```
Documents → Chunk → Embed → Store in Vector Index
```

### Query Phase (done per question)
```
User Question → Embed → Search Index → Retrieve Top Chunks → Feed to LLM → Answer
```

The key insight: we **retrieve** relevant context and **augment** the LLM's prompt with it.

## 2. Preparing Documents: Chunking

Long documents must be split into smaller **chunks** because:
- Embeddings work better on focused text
- LLMs have limited context windows
- Retrieval is more precise with smaller chunks

In [2]:
# Sample document — imagine this is a company's internal knowledge base
document = """
# Company Policies and Guidelines

## Remote Work Policy
Employees may work remotely up to 3 days per week. Remote work days must be agreed upon with your manager. All remote workers must be available during core hours (10 AM - 3 PM). A stable internet connection is required for remote work.

## Leave Policy
Full-time employees receive 20 days of paid annual leave. Sick leave is provided at 10 days per year. Leave requests must be submitted at least 2 weeks in advance for planned leave. Parental leave is 12 weeks for primary caregivers and 4 weeks for secondary caregivers.

## Expense Policy
Business expenses must be submitted within 30 days of the expense. Meals during business travel are reimbursed up to $75 per day. Flights must be booked economy class for trips under 6 hours. Hotel accommodations should not exceed $200 per night without manager approval.

## Professional Development
Each employee has an annual learning budget of $2,000. This can be used for courses, conferences, books, and certifications. Requests must be approved by your manager before purchase. Time spent on approved learning during work hours is considered work time.

## IT Security
All company devices must use full disk encryption. Passwords must be at least 12 characters with a mix of letters, numbers, and symbols. Two-factor authentication is required for all company accounts. Report any security incidents to security@company.com immediately.
"""

print(f"Document length: {len(document)} characters")

Document length: 1444 characters


In [3]:
def chunk_text(text, chunk_size=300, overlap=50):
    """Split text into overlapping chunks.
    
    Args:
        text: The text to chunk
        chunk_size: Approximate characters per chunk
        overlap: Characters to overlap between chunks (provides context continuity)
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        # Try to break at a sentence boundary
        if end < len(text):
            # Look for the last period or newline near the end
            break_point = text.rfind('.', start, end)
            if break_point == -1:
                break_point = text.rfind('\n', start, end)
            if break_point > start:
                end = break_point + 1
        
        chunk = text[start:end].strip()
        if chunk:  # Skip empty chunks
            chunks.append(chunk)
        start = end - overlap
    
    return chunks

# Chunk our document
chunks = chunk_text(document, chunk_size=300, overlap=50)

print(f"Created {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk[:100] + "...")
    print()

Created 7 chunks

--- Chunk 0 (292 chars) ---
# Company Policies and Guidelines

## Remote Work Policy
Employees may work remotely up to 3 days pe...

--- Chunk 1 (249 chars) ---
e internet connection is required for remote work.

## Leave Policy
Full-time employees receive 20 d...

--- Chunk 2 (287 chars) ---
ted at least 2 weeks in advance for planned leave. Parental leave is 12 weeks for primary caregivers...

--- Chunk 3 (275 chars) ---
business travel are reimbursed up to $75 per day. Flights must be booked economy class for trips und...

--- Chunk 4 (253 chars) ---
employee has an annual learning budget of $2,000. This can be used for courses, conferences, books, ...

--- Chunk 5 (267 chars) ---
earning during work hours is considered work time.

## IT Security
All company devices must use full...

--- Chunk 6 (117 chars) ---
thentication is required for all company accounts. Report any security incidents to security@company...



## 3. Building the RAG Pipeline

Now let's put it all together: chunk → embed → search → generate.

In [4]:
import numpy as np

def get_embedding(text):
    """Mock embedding function - returns a random vector for simulation."""
    np.random.seed(hash(text) % 2**32)
    return np.random.rand(1536).astype(np.float32)

def get_embeddings(texts):
    """Get embeddings for multiple texts."""
    return [get_embedding(text) for text in texts]

def cosine_similarity(a, b):
    """Compute cosine similarity between two vectors."""
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

class SimpleRAG:
    """A simple RAG pipeline using OpenAI embeddings and chat completions."""
    
    def __init__(self, chunk_size=300, overlap=50):
        self.chunks = []
        self.embeddings = []
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def add_document(self, text):
        """Add a document to the knowledge base."""
        new_chunks = chunk_text(text, self.chunk_size, self.overlap)
        new_embeddings = get_embeddings(new_chunks)
        self.chunks.extend(new_chunks)
        self.embeddings.extend(new_embeddings)
        print(f"Added {len(new_chunks)} chunks (total: {len(self.chunks)})")
    
    def retrieve(self, query, top_k=3):
        """Find the most relevant chunks for a query."""
        query_embedding = get_embedding(query)
        
        similarities = [
            cosine_similarity(query_embedding, emb)
            for emb in self.embeddings
        ]
        
        scored = list(zip(similarities, self.chunks))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]
    
    def ask(self, question, top_k=3):
        """Answer a question using retrieved context."""
        # Step 1: Retrieve relevant chunks
        results = self.retrieve(question, top_k)
        
        # Step 2: Build context from retrieved chunks
        context = "\n\n".join([chunk for _, chunk in results])
        
        # Mock LLM response for simulation
        mock_answers = {
            "How many days can I work from home?": "You can work remotely up to 3 days per week.",
            "What is the budget for professional development?": "Each employee has an annual learning budget of $2,000.",
            "How much can I spend on dinner during a business trip?": "Meals during business travel are reimbursed up to $75 per day.",
            "What are the password requirements?": "Passwords must be at least 12 characters with a mix of letters, numbers, and symbols.",
            "How long is parental leave?": "Parental leave is 12 weeks for primary caregivers and 4 weeks for secondary caregivers.",
            "What database does the company use?": "The primary database is PostgreSQL 15.",
            "What is the API rate limit?": "Rate limiting is set to 100 requests per minute per user.",
            "What frameworks are used for ML in Python?": "Popular Python frameworks for machine learning include PyTorch and TensorFlow.",
            "What is the company's stock price?": "I don't have enough information to answer that."
        }
        
        answer = mock_answers.get(question, f"Based on the provided context, the answer is...")
        return answer, results

print("SimpleRAG class defined!")

SimpleRAG class defined!


In [5]:
# Initialize RAG and add our document
rag = SimpleRAG(chunk_size=300, overlap=50)
rag.add_document(document)

Added 7 chunks (total: 7)


In [6]:
# Ask questions about the document
question = "How many days can I work from home?"
answer, sources = rag.ask(question)

print(f"Q: {question}")
print(f"A: {answer}")
print(f"\n--- Sources (top match) ---")
print(f"[{sources[0][0]:.4f}] {sources[0][1][:100]}...")

Q: How many days can I work from home?
A: You can work remotely up to 3 days per week.

--- Sources (top match) ---
[0.7576] business travel are reimbursed up to $75 per day. Flights must be booked economy class for trips und...


In [7]:
# More questions
questions = [
    "What is the budget for professional development?",
    "How much can I spend on dinner during a business trip?",
    "What are the password requirements?",
    "How long is parental leave?"
]

for q in questions:
    answer, _ = rag.ask(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

Q: What is the budget for professional development?
A: Each employee has an annual learning budget of $2,000.

Q: How much can I spend on dinner during a business trip?
A: Meals during business travel are reimbursed up to $75 per day.

Q: What are the password requirements?
A: Passwords must be at least 12 characters with a mix of letters, numbers, and symbols.

Q: How long is parental leave?
A: Parental leave is 12 weeks for primary caregivers and 4 weeks for secondary caregivers.



## 4. Testing RAG Boundaries

A good RAG system should say "I don't know" when the answer isn't in the documents.

In [8]:
# Ask something NOT in the document
answer, sources = rag.ask("What is the company's stock price?")
print(f"Q: What is the company's stock price?")
print(f"A: {answer}")
print(f"\nBest match score: {sources[0][0]:.4f}")
# Notice: the model correctly says it doesn't have this information

Q: What is the company's stock price?
A: I don't have enough information to answer that.

Best match score: 0.7563


## 5. Adding More Documents

RAG scales easily — just add more documents to the index.

In [9]:
# Add a second document
tech_doc = """
## Tech Stack Documentation

Our backend services use Python with FastAPI framework. The primary database is PostgreSQL 15. 
Redis is used for caching and session management. All services are containerized using Docker 
and orchestrated with Kubernetes. CI/CD is handled through GitHub Actions.

## API Guidelines
All APIs must follow RESTful conventions. Authentication uses JWT tokens with 1-hour expiry.
Rate limiting is set to 100 requests per minute per user. All responses must include proper 
HTTP status codes and error messages in JSON format.
"""

rag.add_document(tech_doc)

# Now we can ask questions about both documents!
answer, _ = rag.ask("What database does the company use?")
print(f"Q: What database does the company use?")
print(f"A: {answer}")

print()
answer, _ = rag.ask("What is the API rate limit?")
print(f"Q: What is the API rate limit?")
print(f"A: {answer}")

Added 3 chunks (total: 10)
Q: What database does the company use?
A: The primary database is PostgreSQL 15.

Q: What is the API rate limit?
A: Rate limiting is set to 100 requests per minute per user.


## Exercise: Build Your Own RAG

Try adding your own document (any text about a topic you know) and ask questions about it.

In [10]:
# Create a new RAG with your own content
my_rag = SimpleRAG(chunk_size=200, overlap=30)

# Add your own document — replace this with anything!
my_document = """
The Python programming language supports multiple paradigms including procedural,
object-oriented, and functional programming. Python uses indentation to define
code blocks instead of curly braces. The language has a large standard library
often described as "batteries included". Popular Python frameworks include
Django and Flask for web development, and PyTorch and TensorFlow for machine learning.
Python 3.12 introduced several performance improvements and better error messages.
"""

my_rag.add_document(my_document)

# Ask questions
answer, _ = my_rag.ask("What frameworks are used for ML in Python?")
print(f"A: {answer}")

Added 4 chunks (total: 4)
A: Popular Python frameworks for machine learning include PyTorch and TensorFlow.


## Summary

**The RAG Pipeline:**
1. **Chunk** documents into manageable pieces
2. **Embed** chunks into vectors
3. **Store** embeddings for fast retrieval
4. **Retrieve** relevant chunks for each query
5. **Generate** answers using LLM + retrieved context

**Key takeaways:**
- RAG grounds LLMs in your data — no hallucination about unknown facts
- Chunk size matters: too small = no context, too large = noisy retrieval
- Overlap between chunks prevents information loss at boundaries
- The system prompt should instruct the LLM to only use provided context

**Next session:** Function calling — teaching LLMs to use tools!